# M08 — Predict a Continuous Outcome

Build and diagnose a complete regression system on a deterministic local dataset.

**Whole-first path:** baseline → train/test split → fit → predict → metrics → residuals → diagnose.

The goal is trustworthy generalization evidence, not the largest possible score.

## 1. Mission contract and prediction boundary

At prediction time, the system estimates `sale_price_k` (thousands of currency units) immediately before a transaction closes. Only facts already known at that moment are admissible.

`post_sale_assessment_k` is deliberately present as a controlled leakage fixture. It is created after the sale and must never enter the safe model.

Runtime: CPU only, no secrets, no paid API and no network dependency. The bundled data is synthetic and cannot establish real-market validity.

In [ ]:
from pathlib import Path
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

RANDOM_STATE = 42
TARGET = "sale_price_k"
IDENTIFIER = "transaction_id"
LEAKAGE_FEATURE = "post_sale_assessment_k"
SAFE_FEATURES = [
    "floor_area_m2",
    "bedrooms",
    "building_age_years",
    "distance_to_transit_km",
    "neighborhood_score",
    "renovation_quality",
    "energy_efficiency",
    "local_job_access_score",
]

In [ ]:
def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "datasets" / "M08" / "housing_regression.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the LearningOS-AI repository.")


ROOT = find_repository_root(Path.cwd().resolve())
DATA_PATH = ROOT / "datasets" / "M08" / "housing_regression.csv"
data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(data)} rows from {DATA_PATH.relative_to(ROOT)}")
display(data.head(3))

In [ ]:
expected_columns = {IDENTIFIER, TARGET, LEAKAGE_FEATURE, *SAFE_FEATURES}
assert set(data.columns) == expected_columns
assert len(data) == 320
assert data[IDENTIFIER].is_unique
assert not data[list(expected_columns - {IDENTIFIER})].isna().any().any()
assert data[TARGET].dtype.kind in "fiu"
assert data[TARGET].nunique() > 300

data_summary = data[[*SAFE_FEATURES, TARGET]].describe().T[["mean", "std", "min", "max"]]
display(data_summary.round(2))

### Availability boundary

The identifier is excluded because it has no intended predictive meaning. The target is excluded because it is the unknown outcome. The post-sale assessment is excluded because it does not exist at prediction time.

This boundary is a domain contract. Correlation, a random split and cross-validation cannot decide it for us.

In [ ]:
system_map = pd.DataFrame(
    [
        ("input", "SAFE_FEATURES", "known before close"),
        ("target", TARGET, "observed at close"),
        ("forbidden fixture", LEAKAGE_FEATURE, "created after close"),
        ("learned state", "RandomForestRegressor", "fit on training rows only"),
        ("output", "predicted sale_price_k", "one value per held-out row"),
        ("diagnostics", "metrics + residuals + CV", "evidence, not deployment proof"),
    ],
    columns=["role", "artifact", "boundary"],
)
display(system_map)

### Code reading before experimentation

Trace `data → SAFE_FEATURES → split → fit → predict → residuals → metrics`. Before running further, identify which rows can change fitted model state and why `actual - predicted` produces a positive residual for under-prediction.

## 2. Baseline

The baseline policy always predicts the **training-target mean**. It supplies the minimum useful reference: a complex model that cannot beat it has not extracted useful generalizable signal.

We define the policy before splitting, but it will learn the mean from training rows only.

In [ ]:
BASELINE_STRATEGY = "mean"
baseline = DummyRegressor(strategy=BASELINE_STRATEGY)
print("Baseline policy:", BASELINE_STRATEGY)

## 3. Train/test split

The test partition is the final held-out generalization check. Candidate diagnosis and cross-validation will use the training partition only.

In [ ]:
train_rows, test_rows = train_test_split(
    data,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

X_train = train_rows[SAFE_FEATURES]
y_train = train_rows[TARGET]
X_test = test_rows[SAFE_FEATURES]
y_test = test_rows[TARGET]

print({"training_rows": len(train_rows), "test_rows": len(test_rows)})

In [ ]:
assert set(train_rows.index).isdisjoint(test_rows.index)
assert len(train_rows) + len(test_rows) == len(data)
assert list(X_train.columns) == SAFE_FEATURES
assert LEAKAGE_FEATURE not in X_train.columns
assert TARGET not in X_train.columns
print("Split integrity and safe feature allow-list: PASS")

## 4. Fit

The main candidate is a capacity-controlled random forest. `min_samples_leaf=3` reduces leaf memorization; a fixed seed and single CPU worker make execution repeatable where practical.

### Prediction before action — E1: beat the baseline

Write before running: Will the safe model reduce held-out MAE by at least 30% relative to the mean baseline? Which target ranges may remain difficult?

> Your prediction and reason: ________________________________________________

In [ ]:
safe_model = Pipeline(
    steps=[
        (
            "model",
            RandomForestRegressor(
                n_estimators=160,
                min_samples_leaf=3,
                max_features=0.8,
                random_state=RANDOM_STATE,
                n_jobs=1,
            ),
        )
    ]
)

fit_started = time.perf_counter()
baseline.fit(X_train, y_train)
safe_model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - fit_started
print(f"Fitted baseline and safe model in {fit_seconds:.3f} seconds")

## 5. Predict

Both estimators predict the same untouched test rows. Keeping row and target alignment explicit prevents metrics from comparing different observations.

In [ ]:
baseline_predictions = baseline.predict(X_test)
safe_predictions = safe_model.predict(X_test)

assert baseline_predictions.shape == safe_predictions.shape == y_test.shape
prediction_preview = pd.DataFrame(
    {
        "actual": y_test.to_numpy(),
        "baseline_prediction": baseline_predictions,
        "safe_prediction": safe_predictions,
    },
    index=y_test.index,
)
display(prediction_preview.head().round(2))

## 6. Metrics

- **MAE:** average absolute miss, in thousands; easy to explain.
- **RMSE:** also in thousands, with extra weight on large misses.
- **R²:** fraction of target variance explained relative to a mean prediction; unitless and not an error magnitude.

No metric establishes causality, fairness or deployment readiness.

In [ ]:
def regression_metrics(actual, predicted) -> dict[str, float]:
    return {
        "MAE_k": float(mean_absolute_error(actual, predicted)),
        "RMSE_k": float(root_mean_squared_error(actual, predicted)),
        "R2": float(r2_score(actual, predicted)),
    }


baseline_metrics = regression_metrics(y_test, baseline_predictions)
safe_metrics = regression_metrics(y_test, safe_predictions)
metrics_table = pd.DataFrame(
    [baseline_metrics, safe_metrics],
    index=["training-mean baseline", "safe random forest"],
)
display(metrics_table.round(3))

In [ ]:
mae_improvement = 1.0 - safe_metrics["MAE_k"] / baseline_metrics["MAE_k"]
assert safe_metrics["MAE_k"] < baseline_metrics["MAE_k"]
assert safe_metrics["RMSE_k"] >= safe_metrics["MAE_k"]
assert math.isfinite(safe_metrics["R2"])
print(f"Held-out MAE reduction versus baseline: {mae_improvement:.1%}")

### Interpret, do not merely report

Record whether E1 matched your prediction. Translate MAE and RMSE back into `sale_price_k` units. Explain why RMSE exceeding MAE indicates sensitivity to larger misses and why a positive R² can coexist with operationally unacceptable errors.

## 7. Residuals

We define `residual = actual - predicted`. Positive values mean the model under-predicted; negative values mean it over-predicted. A trustworthy diagnosis checks both the residual distribution and structure against fitted values.

### Prediction before action — E2: residual behavior

Write before running: Will errors be centered near zero? Will the spread grow for higher-priced homes, and which visible pattern would weaken trust?

> Your prediction and reason: ________________________________________________

In [ ]:
residuals = y_test.to_numpy() - safe_predictions

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(y_test, safe_predictions, alpha=0.72, edgecolor="none")
limits = [float(min(y_test.min(), safe_predictions.min())), float(max(y_test.max(), safe_predictions.max()))]
axes[0].plot(limits, limits, linestyle="--", color="black", linewidth=1)
axes[0].set(xlabel="Actual sale_price_k", ylabel="Predicted sale_price_k", title="Predicted versus actual")

axes[1].scatter(safe_predictions, residuals, alpha=0.72, edgecolor="none")
axes[1].axhline(0.0, linestyle="--", color="black", linewidth=1)
axes[1].set(xlabel="Predicted sale_price_k", ylabel="Residual (actual - predicted)", title="Held-out residuals")
fig.tight_layout()
plt.show()

residual_summary = pd.Series(residuals).describe(percentiles=[0.1, 0.5, 0.9])
display(residual_summary.to_frame("residual_k").round(2))

### Prediction before action — residual slices

Write before running: In which predicted-price band will absolute error be largest? A band with few cases should produce more or less confidence?

> Your prediction and reason: ________________________________________________

In [ ]:
residual_frame = test_rows[[IDENTIFIER, "floor_area_m2", TARGET]].copy()
residual_frame["prediction"] = safe_predictions
residual_frame["residual"] = residuals
residual_frame["absolute_error"] = np.abs(residuals)
residual_frame["predicted_price_band"] = pd.qcut(
    residual_frame["prediction"],
    q=4,
    labels=["low", "mid-low", "mid-high", "high"],
)

residual_by_band = (
    residual_frame.groupby("predicted_price_band", observed=True)
    .agg(
        rows=("residual", "size"),
        mean_residual_k=("residual", "mean"),
        mae_k=("absolute_error", "mean"),
    )
)
display(residual_by_band.round(2))

Record one supported observation and one uncertainty. Random scatter around zero supports the mean-function fit; curvature, a funnel shape or a band-specific bias suggests a missing relationship, unequal variance or a weak data slice. A pattern is a hypothesis generator, not proof of its cause.

## 8. Diagnose

The test score is one sample. We now use training-only cross-validation, capacity comparisons and held-out perturbation to understand variability, underfitting, overfitting and model reliance.

### 8.1 Cross-validation — Prediction before action

Write before running: Will five-fold validation MAE be close to the final test MAE? How large a fold-to-fold spread would change your confidence?

> Your prediction and reason: ________________________________________________

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    safe_model,
    X_train,
    y_train,
    cv=cv,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    return_train_score=True,
    n_jobs=1,
)
cv_mae = -cv_results["test_mae"]
cv_r2 = cv_results["test_r2"]
cv_table = pd.DataFrame(
    {
        "fold": np.arange(1, 6),
        "validation_MAE_k": cv_mae,
        "validation_R2": cv_r2,
    }
)
display(cv_table.round(3))
print(f"CV MAE: {cv_mae.mean():.2f} ± {cv_mae.std(ddof=1):.2f} k")
print(f"Final test MAE: {safe_metrics['MAE_k']:.2f} k")

Compare the held-out MAE with both the CV mean and its spread. A single test result inside the fold range is compatible with sampling variability; disagreement calls for split, drift or subgroup investigation. Do not tune against the final test result repeatedly.

### 8.2 Underfitting and overfitting — Prediction before action

Write before running: Which tree will have the smallest training error? Which will have the largest training-to-validation gap? Predict the signature of the depth-1 tree.

> Your prediction and reason: ________________________________________________

In [ ]:
capacity_rows = []
for label, depth in [("depth=1", 1), ("depth=5", 5), ("unrestricted", None)]:
    candidate = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    scores = cross_validate(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=1,
    )
    train_mae = -scores["train_score"]
    validation_mae = -scores["test_score"]
    capacity_rows.append(
        {
            "capacity": label,
            "train_MAE_k": train_mae.mean(),
            "validation_MAE_k": validation_mae.mean(),
            "generalization_gap_k": validation_mae.mean() - train_mae.mean(),
        }
    )

capacity_table = pd.DataFrame(capacity_rows).set_index("capacity")
display(capacity_table.round(2))

In [ ]:
underfit_row = capacity_table.loc["depth=1"]
overfit_row = capacity_table.loc["unrestricted"]
assert underfit_row["train_MAE_k"] > capacity_table["train_MAE_k"].min()
assert overfit_row["generalization_gap_k"] == capacity_table["generalization_gap_k"].max()
print("Depth 1: high train and validation error is the underfitting signature.")
print("Unrestricted: tiny train error plus a large validation gap is the overfitting signature.")

### 8.3 Feature influence (not causal) — Prediction before action

Write before running: Which safe feature will cause the largest held-out MAE increase when shuffled? Which correlated features might share or mask importance?

> Your prediction and reason: ________________________________________________

In [ ]:
permutation = permutation_importance(
    safe_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=15,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
influence = (
    pd.DataFrame(
        {
            "feature": SAFE_FEATURES,
            "mae_increase_mean_k": permutation.importances_mean,
            "mae_increase_std_k": permutation.importances_std,
        }
    )
    .sort_values("mae_increase_mean_k", ascending=False)
    .reset_index(drop=True)
)
display(influence.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
ordered = influence.sort_values("mae_increase_mean_k")
ax.barh(
    ordered["feature"],
    ordered["mae_increase_mean_k"],
    xerr=ordered["mae_increase_std_k"],
    alpha=0.8,
)
ax.set(xlabel="Held-out MAE increase after shuffle (k)", title="Permutation influence with repeat variability")
fig.tight_layout()
plt.show()

Permutation influence is **not causal**. It measures this fitted model's reliance on a feature for these held-out rows. Correlated features can divide or hide importance, small samples create variability, and a feature can be predictively useful while reflecting an unacceptable proxy. Retain the error bars and availability audit.

## 9. Controlled failure: target leakage — Prediction before action

We now violate the contract on purpose by including `post_sale_assessment_k`.

Write before running: Will a random holdout and five-fold CV expose the violation, or will both look deceptively strong? Predict the leaky model's R².

> Your prediction and reason: ________________________________________________

In [ ]:
LEAKY_FEATURES = [*SAFE_FEATURES, LEAKAGE_FEATURE]
leaky_model = Pipeline(steps=[("model", LinearRegression())])
leaky_model.fit(train_rows[LEAKY_FEATURES], y_train)
leaky_predictions = leaky_model.predict(test_rows[LEAKY_FEATURES])
leaky_metrics = regression_metrics(y_test, leaky_predictions)

leaky_cv_results = cross_validate(
    leaky_model,
    train_rows[LEAKY_FEATURES],
    y_train,
    cv=cv,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    n_jobs=1,
)

leakage_comparison = pd.DataFrame(
    [
        {**safe_metrics, "CV_MAE_k": cv_mae.mean()},
        {
            **leaky_metrics,
            "CV_MAE_k": (-leaky_cv_results["test_mae"]).mean(),
        },
    ],
    index=["safe model", "INVALID leaky model"],
)
display(leakage_comparison.round(3))

### Diagnose and repair with an availability audit

The split has no overlapping rows, yet the invalid model looks excellent. The causal mechanism is not row duplication; the feature itself encodes information created after the target event.

In [ ]:
FEATURE_AVAILABILITY = {
    **{feature: "prediction_time" for feature in SAFE_FEATURES},
    LEAKAGE_FEATURE: "post_outcome",
    TARGET: "outcome",
    IDENTIFIER: "identifier_only",
}

invalid_deployment_features = [
    feature
    for feature in LEAKY_FEATURES
    if FEATURE_AVAILABILITY.get(feature) != "prediction_time"
]
REPAIRED_FEATURES = [
    feature
    for feature in LEAKY_FEATURES
    if FEATURE_AVAILABILITY.get(feature) == "prediction_time"
]

assert invalid_deployment_features == [LEAKAGE_FEATURE]
assert REPAIRED_FEATURES == SAFE_FEATURES
assert set(train_rows.index).isdisjoint(test_rows.index)
print("Rejected after availability audit:", invalid_deployment_features)
print("Repaired allow-list:", REPAIRED_FEATURES)

Random holdout and ordinary cross-validation preserve the post-outcome relationship in every partition, so neither automatically detects this leakage. The repair is an explicit prediction-time allow-list backed by schema tests. The lower safe score is the trustworthy one.

## 10. Controlled failure: invalid training-only evaluation — Prediction before action

An unrestricted decision tree can memorize training rows. Predict its training R² and then predict whether five-fold validation MAE will agree with its training MAE.

> Your prediction and reason: ________________________________________________

In [ ]:
memorizing_tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
memorizing_tree.fit(X_train, y_train)
training_predictions = memorizing_tree.predict(X_train)
invalid_evaluation_metrics = regression_metrics(y_train, training_predictions)

tree_cv = cross_validate(
    memorizing_tree,
    X_train,
    y_train,
    cv=cv,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    n_jobs=1,
)
valid_tree_cv = {
    "CV_MAE_k": float((-tree_cv["test_mae"]).mean()),
    "CV_R2": float(tree_cv["test_r2"].mean()),
}

display(
    pd.DataFrame(
        [
            {"MAE_k": invalid_evaluation_metrics["MAE_k"], "R2": invalid_evaluation_metrics["R2"]},
            {"MAE_k": valid_tree_cv["CV_MAE_k"], "R2": valid_tree_cv["CV_R2"]},
        ],
        index=["INVALID training-row evaluation", "training-only cross-validation"],
    ).round(3)
)

The near-perfect training fit measures memory on already-seen rows. The validation gap is evidence of overfitting. Generalization claims require predictions for rows that did not influence fitted state; repeated CV supports model development, while the final test remains a separate check.

## 11. Decision synthesis and ADR

A consequential model choice needs more than a leaderboard. Assemble the baseline comparison, held-out metrics, CV mean and variability, residual findings, capacity gap, runtime and leakage audit before deciding.

In [ ]:
decision_evidence = pd.Series(
    {
        "baseline_test_MAE_k": baseline_metrics["MAE_k"],
        "safe_model_test_MAE_k": safe_metrics["MAE_k"],
        "safe_model_test_RMSE_k": safe_metrics["RMSE_k"],
        "safe_model_test_R2": safe_metrics["R2"],
        "safe_model_CV_MAE_mean_k": cv_mae.mean(),
        "safe_model_CV_MAE_std_k": cv_mae.std(ddof=1),
        "mean_test_residual_k": residuals.mean(),
        "fit_seconds": fit_seconds,
        "forbidden_features_detected": len(invalid_deployment_features),
    },
    name="observed_value",
)
display(decision_evidence.to_frame().round(3))

### Required ADR

Use `missions/M08/adr_prompt.md`. Record the decision, context, alternatives, observed evidence, trade-offs, revisit conditions and status. The leaky model is an inadmissible alternative regardless of its score. Distinguish a proposed learning baseline from production approval.

### Formal engineering review

Use `missions/M08/review_brief.md`. Present architecture, the evaluation boundary, meaningful artifacts, controlled-failure root cause, ADR, validation results and one unresolved uncertainty. Classify every reviewer comment as accepted, rejected or deferred with written reasoning.

## 12. No-AI Gate and transfer

Complete the fresh-dataset task in `missions/M08/no_ai_gate.md` without AI-generated code or prose. The transfer must include a baseline, held-out MAE/RMSE/R², residual diagnosis, training-only CV, a capacity diagnosis and a feature-availability audit.

In [ ]:
assert safe_metrics["MAE_k"] < baseline_metrics["MAE_k"]
assert len(residuals) == len(test_rows)
assert len(cv_mae) == 5 and np.isfinite(cv_mae).all()
assert LEAKAGE_FEATURE not in SAFE_FEATURES
assert invalid_deployment_features == [LEAKAGE_FEATURE]
assert leaky_metrics["R2"] > safe_metrics["R2"]
assert invalid_evaluation_metrics["MAE_k"] < valid_tree_cv["CV_MAE_k"]
print("M08 executable contract checks: PASS")

## Reflection

Explain in plain language:

1. why beating the baseline is necessary but insufficient;
2. what MAE, RMSE and R² each reveal and conceal;
3. what your residual plot suggests;
4. how train/CV gaps distinguish underfitting from overfitting;
5. why feature influence is not causal;
6. why the controlled leakage fooled both holdout and CV;
7. what evidence would make you revisit the ADR.

Do not mark the mission complete until the no-AI transfer and formal review evidence exist.